# M12 · Notebook 04 · Prompt caching: pagar menos por el mismo prompt

En M10 vimos que la atención calcula, para cada token del prompt, sus claves (K) y valores (V). Al generar, el modelo los guarda para no recalcularlos: es el **KV cache**. Si dos peticiones empiezan **exactamente igual**, el proveedor puede reutilizar ese cálculo en lugar de rehacerlo. Eso es el *prompt caching*: menos latencia y, sobre todo, **input mucho más barato**.

En este lab medimos el ahorro real con la API de OpenAI y aprendemos la regla que lo decide todo: **lo estable primero, lo variable al final**.

## Cómo funciona en OpenAI (familia GPT-6, octubre 2026)

| Concepto | Valor |
|---|---|
| Activación | Automática, sin cambiar código |
| Tamaño mínimo | 1.024 tokens de prompt |
| Coincidencia | Prefijo **idéntico** byte a byte |
| Duración | Al menos 30 min desde el último uso (`prompt_cache_options.ttl = "30m"`) |
| Leer de caché | **10%** del precio de input |
| Escribir en caché | **125%** del precio de input (desde GPT-5.6) |

Precios por millón de tokens ([pricing oficial](https://developers.openai.com/api/docs/pricing)):

| Modelo | Input | Input cacheado | Escritura caché | Output |
|---|---|---|---|---|
| gpt-6-luna | $0.10 | $0.01 | $0.125 | $0.50 |
| gpt-6-sol | $2.00 | $0.20 | $2.50 | $10.00 |
| gpt-6-astra | $10.00 | $1.00 | $12.50 | $50.00 |

> Fíjate en la escritura: **escribir cuesta más que no cachear**. Cachear solo compensa si ese prefijo se reutiliza. Lo vamos a ver con números.

## Setup

In [ ]:
%pip install -qU openai pandas matplotlib

In [ ]:
import os, getpass, time, uuid, datetime
import pandas as pd
from openai import OpenAI

# En local la clave se lee de la variable de entorno; en Colab se pide por pantalla.
api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("Enter your OpenAI API Key: ")
client = OpenAI(api_key=api_key)

MODEL = "gpt-6-luna"

# USD por millón de tokens (octubre 2026)
PRICES = {
    "gpt-6-luna":  {"input": 0.10, "cached": 0.01, "write": 0.125, "output": 0.50},
    "gpt-6-sol":   {"input": 2.00, "cached": 0.20, "write": 2.50,  "output": 10.00},
    "gpt-6-astra": {"input": 10.00, "cached": 1.00, "write": 12.50, "output": 50.00},
}


def cost(usage, model=MODEL):
    """Coste real de una llamada, separando input normal, leído de caché y escrito en caché."""
    p = PRICES[model]
    details = usage.prompt_tokens_details
    cached = details.cached_tokens or 0
    written = getattr(details, "cache_write_tokens", 0) or 0
    normal = usage.prompt_tokens - cached - written
    return (normal * p["input"] + cached * p["cached"] + written * p["write"]
            + usage.completion_tokens * p["output"]) / 1e6


def cost_without_cache(usage, model=MODEL):
    """Lo que habría costado la misma llamada si no existiera la caché."""
    p = PRICES[model]
    return (usage.prompt_tokens * p["input"] + usage.completion_tokens * p["output"]) / 1e6

## El escenario: un asistente de soporte con una base de conocimiento larga

Un caso muy típico: un *system prompt* con instrucciones y documentación (miles de tokens) que se repite en **todas** las peticiones, y una pregunta corta del usuario que cambia cada vez.

`RUN_ID` solo sirve para que tu primera ejecución empiece con la caché vacía (si no, reutilizarías la caché de una ejecución anterior y no verías la escritura).

In [ ]:
RUN_ID = uuid.uuid4().hex[:8]

SECCIONES = {
    "Envíos": "Los pedidos se envían en 24-48 h laborables desde el almacén de Zaragoza. Península: 4,95 € o gratis a partir de 50 €. Baleares: 9,95 €. Canarias, Ceuta y Melilla: consultar, pueden aplicarse aranceles. El número de seguimiento llega por email cuando el pedido sale del almacén.",
    "Devoluciones": "Hay 30 días naturales desde la entrega para devolver cualquier producto sin usar y con su embalaje original. La recogida es gratuita en Península. El reembolso se hace al mismo método de pago en un máximo de 14 días tras recibir el producto. Los productos personalizados no admiten devolución salvo defecto.",
    "Garantía": "Todos los productos tienen 3 años de garantía legal. Para tramitarla hace falta el número de pedido y una foto o vídeo del defecto. Si el producto no se puede reparar, se sustituye por uno igual o equivalente. Los daños por mal uso, caídas o humedad no están cubiertos.",
    "Pagos": "Aceptamos tarjeta, Bizum, PayPal y transferencia. El pago a plazos sin intereses está disponible en 3 meses para pedidos de más de 150 €. Las facturas se descargan desde el área de cliente y se pueden emitir a nombre de empresa indicando el NIF antes de finalizar la compra.",
    "Cuenta": "El área de cliente permite ver pedidos, descargar facturas, cambiar direcciones y gestionar suscripciones. Si olvidas la contraseña, usa 'He olvidado mi contraseña'. Las cuentas inactivas durante 3 años se eliminan previo aviso por email.",
    "Suscripción Premium": "Premium cuesta 39 € al año: envíos gratis sin mínimo, devoluciones ampliadas a 60 días y acceso anticipado a rebajas. Se renueva automáticamente y se puede cancelar en cualquier momento desde el área de cliente; no se reembolsa la parte proporcional.",
}

def base_de_conocimiento(repeticiones=6):
    # Repetimos las secciones con casuística para llegar a varios miles de tokens, como un manual real.
    bloques = []
    for n in range(repeticiones):
        for titulo, texto in SECCIONES.items():
            bloques.append(f"### {titulo} (artículo {n + 1})\n{texto}")
    return "\n\n".join(bloques)

INSTRUCCIONES = """Eres el asistente de soporte de la tienda online ACME.
Responde en español, en un máximo de dos frases, usando SOLO la base de conocimiento.
Si la respuesta no está en la base de conocimiento, dilo y ofrece contactar con un agente."""

SYSTEM_PROMPT = f"{INSTRUCCIONES}\n\n# Base de conocimiento (versión {RUN_ID})\n\n{base_de_conocimiento()}"

PREGUNTAS = [
    "¿Cuánto cuesta el envío a Mallorca?",
    "¿Puedo devolver una taza personalizada?",
    "Se me ha roto el producto al caerse, ¿lo cubre la garantía?",
    "¿Puedo pagar a plazos un pedido de 200 €?",
    "Si cancelo Premium, ¿me devolvéis dinero?",
    "¿Hacéis envíos a Portugal?",
]
print(f"System prompt: {len(SYSTEM_PROMPT):,} caracteres")

## Experimento A · Bien hecho: prefijo estable, pregunta al final

Mandamos las 6 preguntas con el **mismo** system prompt delante. La primera llamada escribe en caché; las siguientes deberían leer de ella.

In [ ]:
def preguntar(system_prompt, pregunta, model=MODEL):
    t0 = time.time()
    r = client.chat.completions.create(
        model=model,
        reasoning_effort="none",
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": pregunta}],
    )
    d = r.usage.prompt_tokens_details
    return {
        "pregunta": pregunta[:45],
        "prompt_tokens": r.usage.prompt_tokens,
        "escritos_cache": getattr(d, "cache_write_tokens", 0) or 0,
        "leidos_cache": d.cached_tokens or 0,
        "latencia_s": round(time.time() - t0, 2),
        "coste_$": cost(r.usage, model),
        "coste_sin_cache_$": cost_without_cache(r.usage, model),
        "respuesta": r.choices[0].message.content,
    }

filas_a = [preguntar(SYSTEM_PROMPT, q) for q in PREGUNTAS]
df_a = pd.DataFrame(filas_a)
df_a.drop(columns="respuesta")

In [ ]:
for f in filas_a:
    print(f"- {f['pregunta']}\n  {f['respuesta']}\n")

## Experimento B · Mal hecho: algo variable al principio

Error muy común: poner la fecha, el nombre del usuario o un ID de sesión **al principio** del system prompt. Basta un carácter distinto al inicio para que el prefijo ya no coincida y **nada** se reutilice.

In [ ]:
filas_b = []
for q in PREGUNTAS:
    ahora = datetime.datetime.now().isoformat()  # cambia en cada llamada
    system_mal = f"Fecha y hora actual: {ahora}\n\n{SYSTEM_PROMPT}"
    filas_b.append(preguntar(system_mal, q))
df_b = pd.DataFrame(filas_b)
df_b.drop(columns="respuesta")

La solución es trivial: mover la fecha (o cualquier dato variable) **al final**, por ejemplo dentro del mensaje del usuario. El prefijo largo vuelve a ser idéntico y la caché funciona.

## Comparativa

In [ ]:
resumen = pd.DataFrame({
    "estrategia": ["A · estable primero", "B · variable primero"],
    "tokens_leidos_cache": [df_a.leidos_cache.sum(), df_b.leidos_cache.sum()],
    "coste_real_$": [df_a["coste_$"].sum(), df_b["coste_$"].sum()],
    "coste_sin_cache_$": [df_a["coste_sin_cache_$"].sum(), df_b["coste_sin_cache_$"].sum()],
    "latencia_media_s": [df_a.latencia_s.mean(), df_b.latencia_s.mean()],
})
resumen["ahorro_vs_sin_cache"] = 1 - resumen["coste_real_$"] / resumen["coste_sin_cache_$"]
resumen.style.format({"coste_real_$": "{:.6f}", "coste_sin_cache_$": "{:.6f}",
                      "latencia_media_s": "{:.2f}", "ahorro_vs_sin_cache": "{:+.0%}"})

Lee la última columna con calma:

- **A** ahorra mucho: a partir de la segunda llamada el prefijo cuesta el 10%.
- **B** sale **más caro que no tener caché**: cada llamada escribe un prefijo nuevo a 1,25× y nunca lo reutiliza.

Sobre la columna `latencia_media_s`: con un prefijo de unos 2.600 tokens la diferencia entre A y B es **ruido** (red, carga del servidor, longitud de la respuesta). Procesar 2.600 tokens de input es rapidísimo con o sin caché. La mejora de latencia se nota de verdad con prefijos de **decenas de miles** de tokens (documentos largos, historiales de agente), donde ahorrarse el *prefill* recorta segundos al tiempo hasta el primer token. Aquí la caché es, sobre todo, una cuestión de **coste**.


### A escala

Por llamada son fracciones de céntimo. Proyectamos a un asistente con **100.000 consultas al día** durante un mes, en régimen estable (caché caliente). `gpt-6-sol` cuesta exactamente 20 veces `gpt-6-luna` en todos los conceptos.

In [ ]:
consultas_mes = 100_000 * 30
por_llamada_con = df_a["coste_$"].iloc[1:].mean()        # desde la 2ª llamada: caché caliente
por_llamada_sin = df_a["coste_sin_cache_$"].mean()
for modelo, factor in [("gpt-6-luna", 1), ("gpt-6-sol", 20)]:
    sin = por_llamada_sin * consultas_mes * factor
    con = por_llamada_con * consultas_mes * factor
    print(f"{modelo:11s}  sin caché: {sin:>9,.0f} $/mes   con caché: {con:>9,.0f} $/mes   ahorro: {sin - con:>9,.0f} $/mes")

## ¿Cuándo compensa? El punto de equilibrio

Para un prefijo de `P` tokens reutilizado en `N` llamadas:

- Sin caché: `N · P · precio_input`
- Con caché: `1,25 · P · precio_input` (primera escritura) `+ (N − 1) · 0,10 · P · precio_input`

Con 2 llamadas ya compensa (1,35 frente a 2). Con 100 llamadas pagas aproximadamente el **11%** del input.

In [ ]:
import matplotlib.pyplot as plt

N = range(1, 51)
sin_cache = [n * 1.0 for n in N]
con_cache = [1.25 + (n - 1) * 0.10 for n in N]
plt.figure(figsize=(7, 3.5))
plt.plot(N, sin_cache, label="sin caché")
plt.plot(N, con_cache, label="con caché (escritura 1,25× · lectura 0,10×)")
plt.xlabel("llamadas que reutilizan el mismo prefijo")
plt.ylabel("coste del prefijo (× 1 llamada)")
plt.legend(); plt.grid(alpha=.3); plt.show()

## Buenas prácticas

1. **Estable primero, variable al final**: instrucciones, herramientas, ejemplos few-shot y documentos fijos arriba; pregunta, fecha y datos del usuario abajo.
2. **No reordenes** herramientas ni ejemplos entre llamadas: cambia el prefijo.
3. Usa `prompt_cache_key` para separar la contabilidad de caché por cliente o espacio de trabajo.
4. Mide siempre `usage.prompt_tokens_details.cached_tokens`: es la única prueba de que la caché funciona.
5. Si el tráfico es esporádico (menos de una llamada cada 30 min con el mismo prefijo), la caché caduca y solo pagas escrituras.

> Otros proveedores tienen el mismo concepto con matices: Anthropic usa marcas explícitas `cache_control`, Gemini tiene *context caching* explícito, y los modelos de Ollama Cloud también devuelven `cached_tokens` en `usage`.

## Ejercicio

1. Repite el experimento A con `gpt-6-sol` (añade `model="gpt-6-sol"`). ¿Cambia el porcentaje de ahorro? ¿Y el ahorro en dólares?
2. Reduce `base_de_conocimiento(repeticiones=1)`. ¿Sigue habiendo caché? ¿Por qué? (pista: mínimo de 1.024 tokens)
3. Arregla el experimento B moviendo la fecha al mensaje del usuario y comprueba que vuelve a cachear.